In [1]:
#pip install pyodbc

In [2]:
import pandas as pd
import pyodbc
import os                       # >> importando a biblioteca para manipulação de arquivos e diretórios
from dotenv import load_dotenv  # >> importando a função para carregar as variáveis de ambiente do arquivo .env


In [3]:
caminho_do_arquivo_env = r"C:\Users\Administrador\Documents\00-Projetos\Projeto_Python\Caminhos\caminho.env" #Caminho Completo
load_dotenv(dotenv_path=caminho_do_arquivo_env)

server = os.getenv('Nome_Servidor')
database = os.getenv('Nome_Banco')
conexaoDB = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};' \
                            f'SERVER={server};'
                            f'DATABASE={database};'
                            'Trusted_Connection=yes;')
cursor = conexaoDB.cursor()

In [4]:
Produto = pd.read_excel(os.getenv('Tb_Produto'))
Categoria = pd.read_excel(os.getenv('Tb_Categoria'))
Itens = pd.read_excel(os.getenv('Tb_Itens'))
Ordens = pd.read_excel(os.getenv('Tb_Ordens'))
Clientes = pd.read_csv(os.getenv('CLIENTE_CSV'), delimiter=',')

In [5]:

# Deletando dados das Tabelas para evitar duplicidade
cursor.execute(
"""
truncate table [Categoria]
truncate table [Clientes]
truncate table [Itens]
truncate table [Ordens]
truncate table [Produtos]

"""
) 
cursor.commit() 

In [6]:
#Carga Produto
for index, linha in Produto.iterrows():
    cursor.execute("INSERT INTO PRODUTOS (ID, Nome, Price, Id_Category) VALUES (?,?,?,?)", 
                   linha.ID,
                   linha.Name,
                   linha.Price,
                   linha.Id_Category
                   )
cursor.commit()

In [7]:
#Carga Categoria
for index, linha in Categoria.iterrows():
    cursor.execute("INSERT INTO CATEGORIA (ID, Categoria) VALUES (?,?)", 
                   linha.id,
                   linha.name)
cursor.commit()

In [ ]:
#Carga Itens 
Itens['total_price'] = Itens['total_price'].astype(float) # conversao de dados

for index, linha in Itens.iterrows():
    cursor.execute("INSERT INTO ITENS (id, order_id, product_id, quantity, total_price) VALUES (?, ?, ?, ?, ?)", 
                   linha.id,
                   linha.order_id,
                   linha.product_id,
                   linha.quantity,
                   linha.total_price)
cursor.commit()

In [ ]:
#Carga Ordens
for index, linha in Ordens.iterrows():
    cursor.execute("INSERT INTO ORDENS (id, created_at, customer_id, status) VALUES (?, ?, ?, ?)", 
                   linha.id,
                   linha.created_at,
                   linha.customer_id,
                   linha.status)
cursor.commit() 

In [ ]:
#Carga Clientes
Clientes['created_at'] = pd.to_datetime(Clientes['created_at'])
Clientes['email'] = Clientes['email'].fillna('Sem registro')
Clientes['street'] = Clientes['street'].fillna('Sem Info')
Clientes['number'] = Clientes['number'].fillna('Sem Numero')
Clientes['additionals'] = Clientes['additionals'].fillna('Sem Info')

for index, linha in Clientes.iterrows():   
    linha.email = str(linha.email)  # Converter para o tipo 'str' antes da inserção
    linha.country = str(linha.country)
    linha.state = str(linha.state)
    linha.street = str(linha.street)
    linha.number = str(linha.number)
    linha.additionals = str(linha.additionals)
    
    #linha.cell_phone = str(linha.cell_phone)
    cursor.execute("INSERT INTO CLIENTES (id, created_at,first_name, last_name,email,cell_phone,country, state,street, number, additionals) VALUES (?,?,?,?,?,?,?,?,?,?,?)", 
                   linha.id,
                   linha.created_at,
                   linha.first_name,
                   linha.last_name,
                   linha.email,
                   linha.cell_phone,
                   linha.country,
                   linha.state,
                   linha.street,
                   linha.number,
                   linha.additionals)
cursor.commit()

In [ ]:
#Fechando o Cursos
cursor.close()    #Fechar Cursor
conexaoDB.close() #Fechar Conexao